<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-19</br>
</div>

</br>

In [ ]:
# TODO 0: 실습을 위해 아래 패키지를 import 해주세요.
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import ViTImageProcessor, ViTForImageClassification, pipeline
from datasets import load_dataset
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")

</br>

# 학습 내용
>이번 장에서는 <strong>Vision Transformer(ViT) 모델 활용</strong>에 대해 학습합니다.</br></br>
>ViT의 패치 기반 구조를 이해하고 Hugging Face에서 모델을 로드하여 학습해봅시다.</br></br>

</br>

# Vision Transformer (ViT)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이미지를 패치(Patch)로 분할</mark>하여 Transformer 구조로 처리하는 비전 모델입니다.</br></br>
> CNN 없이도 이미지 분류에서 높은 성능을 달성합니다.</br></br>

CNN은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">지역적 수용장(Local Receptive Field)</mark>, 즉 커널 크기 내의 픽셀 관계만 학습하므로, 이미지의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">멀리 떨어진 영역 간의 전역 관계</mark>를 파악하려면 여러 층을 쌓아야 하는 구조적 한계가 있습니다. ViT는 Transformer의 **Self-Attention** 메커니즘을 이미지에 적용하여 이 문제를 해결합니다. 이미지를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">16×16 크기의 패치로 분할</mark>한 뒤 각 패치를 NLP의 토큰처럼 시퀀스로 처리하면, 한 번의 Attention 연산으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모든 패치 간의 전역 관계</mark>를 직접 포착할 수 있습니다.</br>

이 장을 이해하기 위해서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Self-Attention</mark>(시퀀스 내 모든 요소 쌍의 관계를 가중합으로 계산하는 메커니즘), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Transformer 구조</mark>(Multi-Head Attention + Feed-Forward 블록), CNN 기본 개념(합성곱 연산, 풀링, 특성 맵), 그리고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">패치 임베딩</mark>(이미지 패치를 고정 차원의 벡터로 선형 투영하는 과정)에 대한 배경지식이 도움이 됩니다.</br></br>

</br>

## 패치 임베딩 (Patch Embedding)
> 224x224 이미지를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">16x16 크기의 패치로 분할</mark>하면 14x14 = 196개의 패치가 생성됩니다.</br></br>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">항목</th>
      <th>값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">입력 이미지</td><td>224 x 224</td></tr>
    <tr><td style="text-align:center">패치 크기</td><td>16 x 16</td></tr>
    <tr><td style="text-align:center">패치 수</td><td>196개</td></tr>
    <tr><td style="text-align:center">시퀀스 길이</td><td>197 (196 + [CLS])</td></tr>
  </tbody>
</table>

</br>

## ViTImageProcessor
> ViT 모델에 맞는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이미지 전처리</mark>를 자동 수행합니다.</br></br>

In [ ]:
# TODO 1: processor에 ViTImageProcessor를 로드해봅시다. ("google/vit-base-patch16-224")
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

In [ ]:
# TODO 2: CIFAR-10 데이터셋에서 테스트 이미지를 한 장 가져와봅시다.
cifar10_dataset = load_dataset("cifar10", split="test")
image = cifar10_dataset[0]["img"]
image

In [ ]:
# TODO 3: processor로 이미지를 전처리하고 inputs에 저장해봅시다. (return_tensors="pt")
inputs = processor(images=image, return_tensors="pt")

print(f"pixel_values shape: {inputs['pixel_values'].shape}")
print(f"이미지 크기: {processor.size}")

</br>

## ViTForImageClassification
> 분류 헤드가 포함된 ViT 모델입니다.</br></br>

In [ ]:
# TODO 4: model에 ViTForImageClassification을 로드해봅시다. ("google/vit-base-patch16-224")

model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

💡`from_pretrained()`란?
> 다른 사람이 대규모 데이터로 미리 학습해둔 가중치를 불러오는 것입니다.</br></br>
> 직접 처음부터 학습하면 수일~수주가 걸리지만, 사전학습 가중치를 재활용하면 소량의 데이터로도 빠르게 좋은 성능을 얻을 수 있습니다.

In [ ]:
# TODO 5: torch.no_grad() 블록에서 모델에 inputs를 전달하고 outputs에 저장해봅시다.

with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
# TODO 6: logits에서 가장 높은 확률의 클래스 ID를 predicted_class에 저장해봅시다.

predicted_class = outputs.logits.argmax(-1).item()
print(f"예측 클래스 ID: {predicted_class}")

In [ ]:
# TODO 7: model.config.id2label로 레이블 이름을 출력해봅시다.

label = model.config.id2label[predicted_class]
print(f"예측 레이블: {label}")
print(f"전체 클래스 수: {model.config.num_labels}")

</br>

## ViT vs CNN (ResNet)

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교 항목</th>
      <th>ViT</th>
      <th>CNN (ResNet)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">구조</td><td>Transformer</td><td>합성곱 + 잔차</td></tr>
    <tr><td style="text-align:center">입력 처리</td><td>패치 단위</td><td>픽셀 단위</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">수용 영역</mark></td><td>전역 (Self-Attention)</td><td>지역 (커널 크기)</td></tr>
    <tr><td style="text-align:center">데이터 요구량</td><td>많음</td><td>적음</td></tr>
    <tr><td style="text-align:center">대규모 성능</td><td>우수</td><td>보통</td></tr>
  </tbody>
</table>

💡ViT의 강점
> Self-Attention으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이미지의 전역 관계</mark>를 포착합니다.</br>
> CNN은 로컬 패턴에 강하지만, 먼 거리의 관계를 놓칠 수 있습니다.</br></br>

💡작은 데이터셋에서는?
> ViT는 대규모 데이터에서 학습할 때 강점을 발휘합니다.</br>
> 소규모 데이터에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">CNN(ResNet)이 더 적합</mark>할 수 있습니다.</br></br>

</br>

## HuggingFace 데이터셋 로드
> HuggingFace <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">datasets</mark> 라이브러리를 사용하면 CIFAR-10 등 다양한 데이터셋을 간편하게 로드할 수 있습니다.</br></br>

In [ ]:
# TODO 8: HuggingFace datasets로 CIFAR-10 테스트 데이터를 로드해봅시다.

cifar10_dataset = load_dataset("cifar10", split="test")
label_names = cifar10_dataset.features["label"].names

print(f"테스트 데이터: {len(cifar10_dataset)}개")
print(f"클래스: {label_names}")
print(f"샘플 이미지 크기: {cifar10_dataset[0]['img'].size}")

</br>

## ViT 추론 및 시각화

In [ ]:
# TODO 9: vit_processor에 ViTImageProcessor를 로드해봅시다. ("nateraw/vit-base-patch16-224-cifar10")
vit_processor = ViTImageProcessor.from_pretrained("nateraw/vit-base-patch16-224-cifar10")

In [ ]:
# TODO 10: vit_model에 ViTForImageClassification을 로드해봅시다. ("nateraw/vit-base-patch16-224-cifar10")
vit_model = ViTForImageClassification.from_pretrained("nateraw/vit-base-patch16-224-cifar10")

In [ ]:
# TODO 11: 모델을 평가 모드로 전환해봅시다.
vit_model.eval()

In [ ]:
# TODO 12: 5개 샘플에 대해 추론해봅시다.

sample_indices = [0, 100, 200, 300, 400]
sample_images = [cifar10_dataset[i]["img"] for i in sample_indices]
true_labels = [cifar10_dataset[i]["label"] for i in sample_indices]

predictions = []
for img in sample_images:
    # TODO 12-1: vit_processor로 이미지를 전처리해봅시다.
    inputs = vit_processor(images=img, return_tensors="pt")

    # TODO 12-2: torch.no_grad()에서 추론하고 예측 클래스를 predictions에 추가해봅시다.
    with torch.no_grad():
        outputs = vit_model(**inputs)
    predicted_class = outputs.logits.argmax(-1).item()
    predictions.append(predicted_class)

for i, idx in enumerate(sample_indices):
    print(f"샘플 {idx}: 예측={label_names[predictions[i]]}, 실제={label_names[true_labels[i]]}")

In [ ]:
# TODO 13: 추론 결과를 시각화해봅시다.

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i, ax in enumerate(axes):
    ax.imshow(sample_images[i])
    is_correct = predictions[i] == true_labels[i]
    color = "green" if is_correct else "red"
    ax.set_title(f"예측: {label_names[predictions[i]]}\n실제: {label_names[true_labels[i]]}", color=color, fontsize=10)
    ax.axis("off")

plt.suptitle("ViT (CIFAR-10) 추론 결과", fontsize=14)
plt.tight_layout()
plt.show()

💡추론 시 주의사항
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">torch.no_grad()</mark>를 사용하면 그래디언트 계산을 비활성화하여 메모리를 절약할 수 있습니다.</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">model.eval()</mark>을 호출하면 Dropout, BatchNorm 등이 추론 모드로 전환되어 결정적(deterministic) 동작을 보장합니다.</br></br>

</br>

## HuggingFace Pipeline
> HuggingFace <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Pipeline</mark>을 사용하면 전처리부터 추론, 후처리까지 한 줄로 간편하게 수행할 수 있습니다.</br></br>

In [ ]:
# TODO 14: classifier에 image-classification 파이프라인을 생성해봅시다.
classifier = pipeline("image-classification", model="nateraw/vit-base-patch16-224-cifar10")

In [ ]:
# TODO 15: 첫 번째 테스트 이미지로 추론하고 상위 3개 결과를 출력해봅시다.
sample_image = cifar10_dataset[0]["img"]
results = classifier(sample_image)

print("Pipeline 추론 결과:")
for result in results[:3]:
    print(f"  {result['label']}: {result['score']:.4f}")

💡Pipeline의 장점
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">전처리(preprocessing) + 추론(inference) + 후처리(postprocessing)</mark>를 한 줄로 처리할 수 있습니다.</br>
> 빠른 프로토타이핑에 적합하며, 모델과 토크나이저/프로세서를 자동으로 매칭합니다.</br></br>